# 在 AMD Instinct™ GPU 上动手实践 verl Fully Async Policy DAPO 训练

在 [深入 verl Fully Async Training](https://github.com/Vivicai1005/verl-rocm-tutorials/blob/main/verl-fully-async-policy-training-zh.md)  这篇文章中，我们从源码出发介绍了 verl Fully Async Policy 的整体架构，包以及 Rollout、Training 和 Parameter Synchronization 如何协同运行。

理解整体架构后，我们将进一步进入实践，在 AMD Instinct™ GPU 上实际运行一次 Fully Async Policy DAPO Training。

verl 提供了 [dapo_7b_math_fsdp2_4_4.sh](https://github.com/verl-project/verl/blob/main/verl/experimental/fully_async_policy/shell/dapo_7b_math_fsdp2_4_4.sh) 的示例。 该脚本运行在单个 8-GPU 节点上，并将 GPU 资源划分为两个相互独立的部分：
- 4 GPUs for Training: 负责 Actor 的 forward/backward、optimizer update 等训练计算；
- 4 GPUs for Rollout: 运行 vLLM inference engine，持续生成 rollout samples。

Training 和 Rollout 使用相互独立的 GPU 资源，因此 rollout generation 不需要等待每一次 policy update 完成，两条计算流水线可以持续并行运行，这也是 Fully Async Training 的核心特征之一。

在这次实践中，我们将在 AMD Instinct™ GPU 上运行相同架构的 DAPO Training。为了降低硬件资源需求，我们利用 AMD Instinct™ GPU 的大显存，将官方配置中的 **4 Training GPUs + 4 Rollout GPUs** 缩减为 **2 Training GPUs + 2 Rollout GPUs**，仅使用 4 个 GPU 即可完成一次完整的 Fully Async DAPO Training 实践。

接下来，我们将以 dapo_7b_math_fsdp2_4_4.sh 为基础，逐步完成：
1. 了解 DAPO 算法及其相比 GRPO 的关键改进；
2. 验证 AMD ROCm 与 verl 运行环境；
3. 将 Fully Async DAPO Training 配置为 2 个 GPU 用于 Training、2 个 GPU 用于 Rollout，并理解其中的关键参数；
4. 分析训练日志与 metrics，验证 Fully Async Training 是否正常运行。

## 什么是 DAPO 算法，它和 GRPO 有什么区别

[DAPO（Decoupled Clip and Dynamic sAmpling Policy Optimization）](https://arxiv.org/pdf/2503.14476) 是以 GRPO (Group Relative Policy Optimization) 为基础，针对大规模 Long-CoT RL Training 中出现的训练不稳定、entropy collapse、无效样本以及过长 response 等问题进行了一系列的改进。

### GRPO：用同一 Prompt 的多个 Response 计算相对 Advantage
对于一个 prompt，GRPO 不只生成一个 response，而是从当前的 policy 中采样一组 responses，然后分别计算每个 response 的 reward：

$$
R_i = r(q, o_i), \qquad i = 1,2,\dots,G
$$

其中，\(q\) 表示 prompt，\(o_i\) 表示第 \(i\) 个 response，\(R_i\) 表示该 response 对应的 reward。

与标准 PPO 通常需要训练额外的 Critic / Value Model 不同，GRPO 可以直接利用同一 prompt 下多个 responses 的相对 reward 来估计 advantage，而不需要单独训练 Value Model:

$$
\hat{A}_i =
\frac{
R_i - \operatorname{mean}(R_1,\dots,R_G)
}{
\operatorname{std}(R_1,\dots,R_G)
}
$$

一个 response 的 reward 如果高于组内平均水平，就得到正 advantage；低于组内平均水平，则得到负 advantage。计算得到 advantage 之后，GRPO 会用它来更新 policy。对于 response \(o_i\) 中的第 \(t\) 个 token，首先计算新旧 policy 生成该 token 的概率比：

$$
r_{i,t}(\theta)
=
\frac{
\pi_\theta(o_{i,t} \mid q, o_{i,<t})
}{
\pi_{\theta_{\text{old}}}(o_{i,t} \mid q, o_{i,<t})
}
$$
然后使用 advantage 对 policy update 的方向进行加权。简化来看：

$$
L_{i,t} \propto r_{i,t}(\theta)\hat{A}_i
$$

其中，同一个 response 中的 tokens 共享该 response 的 group-relative advantage \($\hat{A}_i\$)。

因此：

- 如果 $\hat{A}_i > 0$，说明这个 response 的 reward 高于组内平均水平，训练会提高生成这个 response 中 tokens 的概率；
- 如果 $\hat{A}_i < 0$，说明这个 response 的 reward 低于组内平均水平，训练会降低生成这些 tokens 的概率；
- 如果 $\hat{A}_i \approx 0$，这个 response 对当前 policy update 基本不会提供有效的训练信号。

实际训练中，GRPO 和 PPO 类似，还会通过 clipping 限制一次 policy update 的幅度：

$$
L_{\text{GRPO}}
=
\frac{1}{G}
\sum_{i=1}^{G}
\frac{1}{|o_i|}
\sum_{t=1}^{|o_i|}
\min
\left(
r_{i,t}(\theta)\hat{A}_i,\,
\operatorname{clip}
\left(
r_{i,t}(\theta),
1-\epsilon,
1+\epsilon
\right)
\hat{A}_i
\right)
$$

### DAPO：在 GRPO 基础上针对 Long-CoT Training 的改进
DAPO 保留了 GRPO 的核心思路：对同一个 prompt 采样多个 responses，根据组内 reward 计算 relative advantage，再使用 advantage 更新 policy。

DAPO 在 GRPO 的基础上针对 Long-CoT RL Training 引入了多项改进。在这里，我们主要介绍当前训练配置中实际使用的几个设计：Clip-Higher、Token-Level Policy Gradient Loss 和 Overlong Reward Shaping。

#### 1. Clip-Higher：增加 Exploration
GRPO 通常使用对称 clipping：

$$
[1-\epsilon,\ 1+\epsilon]
$$

例如 $\epsilon=0.2$ 时：

$$
[0.8,\ 1.2]
$$
这种限制可以避免一次 policy update 过大，但也会限制低概率 token 的提升空间，使 policy 越来越集中在已有 reasoning paths 上，最终可能导致 entropy collapse。

DAPO 将上下 clipping range 解耦：

$$
[1-\epsilon_{\text{low}},\ 1+\epsilon_{\text{high}}]
$$

论文中使用：

$$
\epsilon_{\text{low}}=0.2,\qquad
\epsilon_{\text{high}}=0.28
$$

也就是把 upper bound 从 1.2 提高到 1.28。这样，获得正 advantage 的低概率 tokens 有更大的提升空间，从而鼓励模型探索新的 reasoning paths。

#### 2. Token-Level Policy Gradient Loss：按 Token 聚合 Loss
GRPO 通常先对每个 response 内的 token loss 求平均，再对所有 responses 求平均：
$$
\frac{1}{G}
\sum_{i=1}^{G}
\frac{1}{|o_i|}
\sum_{t=1}^{|o_i|}
L_{i,t}
$$
这意味着无论 response 长短，每个 response 在最终 loss 中的权重基本相同。对于 Long-CoT Training，这会使长 response 中单个 token 的训练信号被稀释。

DAPO 改成 Token-Level Mean：
$$
\frac{
\sum_i \sum_t L_{i,t}
}{
\sum_i |o_i|
}
$$
即直接对整个 batch 中的有效 response tokens 求平均，这样，每个 token 对 policy update 的贡献更加一致，更适合 Long-CoT Training。

#### 3. Overlong Reward Shaping：减少过长 Response 带来的 Reward Noise
Long-CoT response 可能因为达到 max_response_length 而被截断。这种情况下，模型的 reasoning 过程本身可能是合理的，只是还没有来得及输出最终答案。如果直接把这类 truncated response 当作错误结果，并给予较强的 negative reward，就可能引入错误的训练信号。

DAPO 因此使用 Soft Overlong Punishment。它不会等到 response 达到最大长度后再突然施加强惩罚，而是在 response 逐渐接近 max_response_length 时，平滑地增加 length penalty。也就是说，response 长度处于正常范围时不增加额外惩罚；当长度进入预设的 overlong buffer 区间后，response 越长，penalty 越大。

这种设计会在 response 接近最大长度时提前引入平滑的长度惩罚，鼓励模型逐渐学习控制 reasoning length。相比在 response 达到最大长度并被截断时才出现突变的 reward signal，这种方式可以降低 truncation 带来的 reward noise，提高 Long-CoT RL Training 的稳定性。

## 验证 AMD ROCm 与 verl 运行环境
启动训练之前，我们先确认当前容器能够正确识别 AMD GPU，并验证 ROCm、verl 和 vLLM 环境是否可以正常使用。

### 检查 AMD GPU
首先使用 amd-smi 检查容器中可见的 AMD GPU：

In [ ]:
!amd-smi

本次实践使用 4 个 AMD Instinct™ GPU devices。正常情况下，`amd-smi`或者 `rocm-smi` 能够识别并显示全部 4 个 GPU，以及显存占用、温度和设备状态等信息。

### 加载 verl Python 环境并验证依赖
verl container 中已经预配置了 `/opt/venv` Python 环境。下面在当前 shell 中激活该环境，并确认 verl 和 vLLM 可以正常加载。

In [ ]:
%%bash
set -e

source /opt/venv/bin/activate

TUTORIAL_DIR=$(find /workspace -maxdepth 1 -type d \
  -name 'verl-dapo-fully-async-policy-training-*' \
  | head -n 1)

if [ -z "${TUTORIAL_DIR}" ]; then
    echo "Error: tutorial workspace directory not found."
    exit 1
fi

export PYTHONPATH="${TUTORIAL_DIR}/vllm:${PYTHONPATH:-}"

echo "Python:"
which python
python --version

python - <<'PY'
import verl
import vllm

print("verl version:", verl.__version__)
print("verl module:", verl.__file__)
print("vLLM version:", vllm.__version__)
print("vLLM module:", vllm.__file__)
PY

##  准备并配置 Fully Async DAPO Training
接下来，我们先下载本次训练使用的模型并准备 DAPO 数据集，然后将 Fully Async DAPO Training 配置为 2 个 GPU 用于 Training、2 个 GPU 用于 Rollout，并结合启动配置介绍其中的关键训练参数。

### 下载模型
本次实践使用 [Qwen2.5-Math-7B](https://huggingface.co/Qwen/Qwen2.5-Math-7B) 作为基础模型。运行下面的命令，将模型下载到 `/root/verl/models/Qwen2.5-Math-7B`：

In [ ]:
%%bash
set -e

hf download Qwen/Qwen2.5-Math-7B --local-dir /root/verl/models/Qwen2.5-Math-7B

Qwen2.5-Math-7B 默认配置的最大位置编码长度无法覆盖本次训练使用的长序列。将模型下载完成后，把 `config.json` 中的 `max_position_embeddings` 修改为 `32768`：

In [ ]:
%%bash
set -e

MODEL_PATH=/root/verl/models/Qwen2.5-Math-7B

python - <<'PY'
import json
from pathlib import Path

config_path = Path("/root/verl/models/Qwen2.5-Math-7B/config.json")

with config_path.open("r", encoding="utf-8") as f:
    config = json.load(f)

old_value = config.get("max_position_embeddings")
config["max_position_embeddings"] = 32768

with config_path.open("w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)
    f.write("\n")

print(f"Updated max_position_embeddings: {old_value} -> 32768")
PY

### 准备训练与验证数据
下载并运行 verl-recipe 提供的 DAPO 数据准备脚本，该脚本会准备本次训练使用的 DAPO-Math-17k 训练集和 AIME 2024 验证集：

In [ ]:
%%bash
set -e

source /opt/venv/bin/activate

wget -q -O prepare_dapo_data.sh \
    https://raw.githubusercontent.com/verl-project/verl-recipe/main/dapo/prepare_dapo_data.sh

# 让脚本内部的 wget 强制使用单行进度条
wget() {
    command wget --progress=bar:force:noscroll "$@"
}
export -f wget

bash prepare_dapo_data.sh

数据准备完成后，将生成以下两个文件：

- `/root/verl/data/dapo-math-17k.parquet`：训练数据集；
- `/root/verl/data/aime-2024.parquet`：验证数据集。

### 理解 Fully Async DAPO Training 的关键参数
模型与数据准备完成后，我们先介绍启动 Fully Async DAPO Training 时使用的关键参数。

#### 配置模型与数据路径

首先指定模型、训练集、验证集和 checkpoint 的保存路径：

```bash
RAY_DATA_HOME=${RAY_DATA_HOME:-"${HOME}/verl"}

MODEL_PATH=${MODEL_PATH:-"${RAY_DATA_HOME}/models/Qwen2.5-Math-7B"}
CKPTS_DIR=${CKPTS_DIR:-"${RAY_DATA_HOME}/ckpts/${project_name}/${exp_name}"}

TRAIN_FILE=${TRAIN_FILE:-"${RAY_DATA_HOME}/data/dapo-math-17k.parquet"}
TEST_FILE=${TEST_FILE:-"${RAY_DATA_HOME}/data/aime-2024.parquet"}
```

本次训练选用 **Qwen2.5-Math-7B** 作为基础模型，以 **DAPO-Math-17k** 作为训练集，并使用 **AIME 2024** 评估训练过程中的模型表现。

#### Training 与 Rollout 的 GPU 划分
本次训练使用单节点的 4 个 GPU，并将其中 2 个分配给 Rollout，其余 2 个分配给 Training：

```bash
NNODES=${NNODES:-1}
NGPUS_PER_NODE=${NGPUS_PER_NODE:-4}

n_gpus_rollout=${N_GPUS_ROLLOUT:-2}
n_gpus_training=$((NGPUS_PER_NODE - n_gpus_rollout))
```

#### 配置 Rollout Engine
Rollout 侧使用 vLLM 的异步推理模式：

```bash
rollout_mode="async"
rollout_name="vllm"

export VLLM_USE_V1=1
return_raw_chat="True"
```

#### Prompt 与 Response 长度
本次训练的最大 Prompt 和 Response 长度分别为：
```bash
max_prompt_length=$((1024 * 2))
max_response_length=$((1024 * 8))
```

#### 配置 DAPO 算法参数
##### Group Relative Policy Optimization
```bash
adv_estimator=grpo
n_resp_per_prompt=16
```
adv_estimator=grpo 表示使用 GRPO 方式估计 advantage。Rollout Engine 会针对每个 Prompt 生成 16 个 Response，并根据同一组 Response 的 reward 计算相对 advantage。

##### Asymmetric Clipping
```bash
clip_ratio_low=0.2
clip_ratio_high=0.28
```
DAPO 分别设置 clipping 区间的上下界。更宽的上界允许具有正 advantage 的 token 获得更大的概率提升空间，有助于缓解 entropy collapse，并增强训练过程中的策略探索能力。

##### Overlong Reward Shaping
```bash
enable_overlong_buffer=True
overlong_buffer_len=$((1024 * 4))
overlong_penalty_factor=1.0
```
Overlong Reward Shaping 用于惩罚过长的 Response。与超过长度限制后直接给予固定惩罚相比，它会在设定的 buffer 区间内逐步增加惩罚，使长度边界附近的 reward 变化更加平滑。

##### Token-level Loss Aggregation
```bash
loss_agg_mode="token-mean"
```
`token-mean` 表示汇总一个 mini-batch 中所有有效 Response tokens 的 loss，然后按照有效 token 总数取平均，使每个有效 token 对最终 loss 具有相同权重。

#### Fully Async Training 参数
```bash
train_prompt_bsz=0
gen_prompt_bsz=1
train_prompt_mini_bsz=32

staleness_threshold=0.1
trigger_parameter_sync_step=4
require_batches=4
partial_rollout=True
```
这些参数共同控制 Rollouter 生成样本、Trainer 消费样本、Actor 更新以及参数同步的节奏。

Fully Async Training 不使用传统的固定全局 Batch，因此将 `train_prompt_bsz` 设置为 0。`gen_prompt_bsz=1` 表示 Rollouter 每次读取并处理一个 Prompt，通过逐样本生产和消费实现 Streaming Training。

`require_batches=4` 表示 Trainer 每次收集 `4 × 32 = 128` 个 Prompt 后进行一次本地更新。由于每个 Prompt 会生成 16 个 Response，因此一次更新包含 2048 条 Response trajectories。

结合 `trigger_parameter_sync_step=4`，Trainer 每完成 4 次本地更新，也就是处理 512 个 Prompt 后，会将最新的 Actor 参数同步到 Rollout Engine。

在 Fully Async Training 中，Rollouter 和 Trainer 并行运行，因此 Trainer 可能会收到由旧版本 Policy 生成的样本。`staleness_threshold=0.1` 表示允许系统使用一定比例的 stale samples，使 Rollouter 可以适度提前生成数据，从而减少 Trainer 等待样本的时间。该值越小，训练越接近 on-policy；值越大，异步程度越高，但 Training Policy 与生成样本所使用的 Rollout Policy 之间的差异也会增大。

当参数同步开始时，Rollout Engine 中可能仍有尚未生成完成的 Response。启用 Partial Rollout `partial_rollout=True` 后，系统可以暂停这些 rollout 并保留已经生成的部分结果，在参数同步完成后继续生成，从而减少等待长 Response 完成所产生的 pipeline bubble。`partial_rollout` 只有在 `staleness_threshold > 0` 时才会生效。

#### Rollout 采样参数
```bash
temperature=1.0
top_p=1.0
top_k=-1
val_top_p=0.7

actor_rollout_ref.rollout.val_kwargs.top_p=${val_top_p}
actor_rollout_ref.rollout.val_kwargs.do_sample=True
actor_rollout_ref.rollout.val_kwargs.n=1
```
训练阶段使用 `temperature=1.0` 和 `top_p=1.0`，保留较强的采样随机性，使同一个 Prompt 可以生成具有差异的多个 Response，为 GRPO 的组内 reward 比较提供足够的探索空间。
验证阶段使用 top_p=0.7，并且每个 Prompt 只生成一个 Response.

#### Rollout Log Probability
```bash
actor_rollout_ref.rollout.calculate_log_probs=True
```
ully Async Training 中的 Rollout Policy 可能落后于 Training Policy，因此需要由 Rollout Engine 记录生成每个 token 时的 log probability，确保 old_log_prob 与实际生成样本的 Policy 版本一致。

#### FSDP2 与并行配置
```bash
fsdp_size=2
gen_tp=1
sp_size=1

ref_offload=True
actor_offload=False
```
Actor 使用 FSDP2，并通过 `fsdp_size=2` 将模型参数分片到 2 个 Training GPU 上。
Rollout Engine 的 Tensor Parallel Size 为 1，表示每个 vLLM Replica 使用一个 GPU；sp_size=1 表示不启用 Ulysses Sequence Parallelism。

#### Dynamic Batch 与 Token Budget
```bash
use_dynamic_bsz=True

actor_ppo_max_token_len=$(((max_prompt_length + max_response_length) * 2))
infer_ppo_max_token_len=$(((max_prompt_length + max_response_length) * 3))
```
启用 Dynamic Batch 后，系统会根据序列的实际 token 数量动态组织 micro-batch，而不是只按照固定的序列数量划分 Batch，从而减少 padding 带来的无效计算。

#### vLLM 显存与 Prefill 配置
```bash
actor_rollout_ref.rollout.gpu_memory_utilization=0.80
actor_rollout_ref.rollout.enable_chunked_prefill=True
actor_rollout_ref.rollout.max_num_batched_tokens=6144
```
vLLM 最多使用约 80% 的 GPU 显存，并通过 Chunked Prefill 分块处理较长 Prompt。`max_num_batched_tokens=6144` 限制一次 vLLM 调度能够处理的最大 token 数量，用于控制 Rollout 侧的显存峰值。

#### Actor 优化器参数
```bash
actor_rollout_ref.actor.optim.lr=1e-6
actor_rollout_ref.actor.optim.lr_warmup_steps=10
actor_rollout_ref.actor.optim.weight_decay=0.1

actor_rollout_ref.actor.entropy_coeff=0
actor_rollout_ref.actor.grad_clip=1.0
```
Actor 使用 `1e-6` 的学习率、10 个 warmup steps 和 `0.1` 的 weight decay。`entropy_coeff=0` 表示不额外加入 entropy bonus，`grad_clip=1.0` 则用于限制梯度范数，避免梯度过大导致训练不稳定。

#### 训练规模与验证
```bash
total_rollout_steps=$((512 * 100))
test_freq=10

rollout.total_rollout_steps=51200
trainer.total_epochs=10
trainer.test_freq=10
trainer.val_before_train=True
```
本次训练最多生成 `51200` 个 Rollout Prompt samples，并最多遍历训练集 10 个 epoch。正式训练开始前会先在 AIME 2024 上执行一次验证，之后每隔 10 个训练 step 进行一次验证。

#### 日志与 Checkpoint
```bash
trainer.logger=['console','wandb']
trainer.default_local_dir="${CKPTS_DIR}"
trainer.resume_mode=auto
trainer.save_freq=-1
```
训练指标会同时输出到终端和 Weights & Biases（W&B），可以通过 W&B Dashboard 实时查看 reward、loss、Response 长度和异步训练状态等指标。resume_mode=auto 表示启动时自动检查 checkpoint 目录并尝试恢复训练；save_freq=-1 表示不进行周期性的 checkpoint 保存。如果需要保存 checkpoint，可以将 save_freq 修改为正整数，例如设置为 50，表示每隔 50 个训练 step 将 checkpoint 保存到 CKPTS_DIR。

## 启动训练并监控训练指标

### 配置 W&B API Key（可选）
如果希望将训练指标记录到 Weights & Biases，请在下面安全输入 W&B API Key。输入内容不会显示在 Notebook 中；直接回车则只输出 Console 日志。

In [ ]:
import getpass
import os

if os.environ.get("WANDB_API_KEY"):
    print("WANDB_API_KEY is already configured.")
else:
    wandb_api_key = getpass.getpass(
        "Enter WANDB_API_KEY (press Enter to skip): "
    ).strip()

    if wandb_api_key:
        os.environ["WANDB_API_KEY"] = wandb_api_key
        print("WANDB_API_KEY configured. W&B logging will be enabled.")
    else:
        print("WANDB_API_KEY not provided. Console logging will be used.")

### 启动训练

In [ ]:
%%bash
set -euo pipefail

RUN_NAME="dapo_qwen2.5_7b_fully_async_2_2"
LOG_DIR="${HOME}/verl/logs"
LOG_FILE="${LOG_DIR}/${RUN_NAME}.log"
PID_FILE="${LOG_DIR}/${RUN_NAME}.pid"

mkdir -p "${LOG_DIR}"

# 避免重复启动训练
if [[ -f "${PID_FILE}" ]]; then
    existing_pid=$(cat "${PID_FILE}")

    if kill -0 "${existing_pid}" 2>/dev/null; then
        echo "Training is already running."
        echo "PID: ${existing_pid}"
        echo "Log: ${LOG_FILE}"
        exit 1
    else
        echo "Removing stale PID file: ${PID_FILE}"
        rm -f "${PID_FILE}"
    fi
fi

nohup setsid bash -s >"${LOG_FILE}" 2>&1 <<'TRAIN_SCRIPT' &
#!/usr/bin/env bash
set -euo pipefail

# Activate Python virtual environment
source /opt/venv/bin/activate

TUTORIAL_DIR=$(find /workspace -maxdepth 1 -type d \
    -name 'verl-dapo-fully-async-policy-training-*' \
    -print -quit)

if [[ -z "${TUTORIAL_DIR}" ]]; then
    echo "Error: tutorial workspace directory not found."
    exit 1
fi

export PYTHONPATH="${TUTORIAL_DIR}/vllm:${PYTHONPATH:-}"

# Write Python logs immediately
export PYTHONUNBUFFERED=1

# Select logger according to WANDB_API_KEY
if [[ -n "${WANDB_API_KEY:-}" ]]; then
    trainer_logger="['console','wandb']"
    echo "WANDB_API_KEY detected. W&B logging is enabled."
else
    trainer_logger="['console']"
    echo "WANDB_API_KEY not found. Console logging is enabled."
fi

project_name='DAPO'
exp_name='DAPO-Qwen2.5-7b-MATH-fsdp2-fully-async-2-2'

# Paths
RAY_DATA_HOME=${RAY_DATA_HOME:-"${HOME}/verl"}

MODEL_PATH=${MODEL_PATH:-"${RAY_DATA_HOME}/models/Qwen2.5-Math-7B"}
CKPTS_DIR=${CKPTS_DIR:-"${RAY_DATA_HOME}/ckpts/${project_name}/${exp_name}"}
TRAIN_FILE=${TRAIN_FILE:-"${RAY_DATA_HOME}/data/dapo-math-17k.parquet"}
TEST_FILE=${TEST_FILE:-"${RAY_DATA_HOME}/data/aime-2024.parquet"}

# Rollout engine
rollout_mode="async"
rollout_name="vllm"

if [[ "${rollout_mode}" == "async" ]]; then
    export VLLM_USE_V1=1
    return_raw_chat="True"
else
    return_raw_chat="False"
fi

# Algorithm parameters
adv_estimator="grpo"

use_kl_in_reward=False
kl_coef=0.0
use_kl_loss=False
kl_loss_coef=0.0

clip_ratio_low=0.2
clip_ratio_high=0.28

# Prompt and response length
max_prompt_length=$((1024 * 2))
max_response_length=$((1024 * 8))

enable_overlong_buffer=True
overlong_buffer_len=$((1024 * 4))
overlong_penalty_factor=1.0

# Loss aggregation
loss_agg_mode="token-mean"

# Sampling parameters
temperature=1.0
top_p=1.0
top_k=-1
val_top_p=0.7

# Performance parameters
use_dynamic_bsz=True

actor_ppo_max_token_len=$(((max_prompt_length + max_response_length) * 2))
infer_ppo_max_token_len=$(((max_prompt_length + max_response_length) * 3))

ref_offload=True
actor_offload=False

gen_tp=1
sp_size=1
fsdp_size=2

# GPU resource allocation
NNODES=${NNODES:-1}
NGPUS_PER_NODE=${NGPUS_PER_NODE:-4}

n_gpus_rollout=${N_GPUS_ROLLOUT:-2}
n_gpus_training=$((NGPUS_PER_NODE - n_gpus_rollout))

# Fully Async parameters
train_prompt_bsz=0
gen_prompt_bsz=1
n_resp_per_prompt=16
train_prompt_mini_bsz=32

total_rollout_steps=$((512 * 100))
test_freq=10

staleness_threshold=0.1
trigger_parameter_sync_step=4
require_batches=4
partial_rollout=True

echo "Starting Fully Async DAPO Training..."
echo "Project: ${project_name}"
echo "Experiment: ${exp_name}"
echo "Model: ${MODEL_PATH}"
echo "Training GPUs: ${n_gpus_training}"
echo "Rollout GPUs: ${n_gpus_rollout}"
echo "Logger: ${trainer_logger}"

python -m verl.experimental.fully_async_policy.fully_async_main \
    data.train_files="${TRAIN_FILE}" \
    data.val_files="${TEST_FILE}" \
    data.prompt_key=prompt \
    data.truncation='left' \
    data.max_prompt_length=${max_prompt_length} \
    data.max_response_length=${max_response_length} \
    data.train_batch_size=${train_prompt_bsz} \
    data.gen_batch_size=${gen_prompt_bsz} \
    data.return_raw_chat=${return_raw_chat} \
    actor_rollout_ref.rollout.n=${n_resp_per_prompt} \
    algorithm.adv_estimator=${adv_estimator} \
    algorithm.use_kl_in_reward=${use_kl_in_reward} \
    algorithm.kl_ctrl.kl_coef=${kl_coef} \
    actor_rollout_ref.actor.fsdp_config.strategy=fsdp2 \
    critic.strategy=fsdp2 \
    actor_rollout_ref.actor.use_kl_loss=${use_kl_loss} \
    actor_rollout_ref.actor.kl_loss_coef=${kl_loss_coef} \
    actor_rollout_ref.actor.clip_ratio_low=${clip_ratio_low} \
    actor_rollout_ref.actor.clip_ratio_high=${clip_ratio_high} \
    actor_rollout_ref.actor.clip_ratio_c=10.0 \
    actor_rollout_ref.model.use_remove_padding=True \
    actor_rollout_ref.hybrid_engine=False \
    +actor_rollout_ref.model.override_config.max_position_embeddings=32768 \
    actor_rollout_ref.actor.use_dynamic_bsz=${use_dynamic_bsz} \
    actor_rollout_ref.ref.log_prob_use_dynamic_bsz=${use_dynamic_bsz} \
    actor_rollout_ref.rollout.log_prob_use_dynamic_bsz=${use_dynamic_bsz} \
    actor_rollout_ref.actor.ppo_max_token_len_per_gpu=${actor_ppo_max_token_len} \
    actor_rollout_ref.ref.log_prob_max_token_len_per_gpu=${infer_ppo_max_token_len} \
    actor_rollout_ref.rollout.log_prob_max_token_len_per_gpu=${infer_ppo_max_token_len} \
    actor_rollout_ref.model.path="${MODEL_PATH}" \
    actor_rollout_ref.actor.optim.lr=1e-6 \
    actor_rollout_ref.actor.optim.lr_warmup_steps=10 \
    actor_rollout_ref.actor.optim.weight_decay=0.1 \
    actor_rollout_ref.actor.ppo_mini_batch_size=${train_prompt_mini_bsz} \
    actor_rollout_ref.actor.fsdp_config.param_offload=${actor_offload} \
    actor_rollout_ref.actor.fsdp_config.optimizer_offload=${actor_offload} \
    actor_rollout_ref.actor.entropy_coeff=0 \
    actor_rollout_ref.actor.grad_clip=1.0 \
    actor_rollout_ref.actor.loss_agg_mode=${loss_agg_mode} \
    actor_rollout_ref.actor.ulysses_sequence_parallel_size=${sp_size} \
    actor_rollout_ref.rollout.gpu_memory_utilization=0.80 \
    actor_rollout_ref.rollout.tensor_model_parallel_size=${gen_tp} \
    actor_rollout_ref.rollout.enable_chunked_prefill=True \
    actor_rollout_ref.rollout.max_num_batched_tokens=$((max_prompt_length + max_response_length)) \
    actor_rollout_ref.rollout.temperature=${temperature} \
    actor_rollout_ref.rollout.top_p=${top_p} \
    actor_rollout_ref.rollout.top_k=${top_k} \
    actor_rollout_ref.rollout.val_kwargs.temperature=${temperature} \
    actor_rollout_ref.rollout.val_kwargs.top_p=${val_top_p} \
    actor_rollout_ref.rollout.val_kwargs.top_k=${top_k} \
    actor_rollout_ref.rollout.val_kwargs.do_sample=True \
    actor_rollout_ref.rollout.val_kwargs.n=1 \
    actor_rollout_ref.rollout.calculate_log_probs=True \
    actor_rollout_ref.ref.fsdp_config.param_offload=${ref_offload} \
    actor_rollout_ref.ref.ulysses_sequence_parallel_size=${sp_size} \
    actor_rollout_ref.actor.fsdp_config.fsdp_size=${fsdp_size} \
    actor_rollout_ref.rollout.name=${rollout_name} \
    actor_rollout_ref.rollout.mode=${rollout_mode} \
    reward.reward_manager.name=dapo \
    +reward.reward_kwargs.overlong_buffer_cfg.enable=${enable_overlong_buffer} \
    +reward.reward_kwargs.overlong_buffer_cfg.len=${overlong_buffer_len} \
    +reward.reward_kwargs.overlong_buffer_cfg.penalty_factor=${overlong_penalty_factor} \
    +reward.reward_kwargs.overlong_buffer_cfg.log=False \
    +reward.reward_kwargs.max_resp_len=${max_response_length} \
    trainer.logger="${trainer_logger}" \
    trainer.project_name="${project_name}" \
    trainer.experiment_name="${exp_name}" \
    trainer.val_before_train=True \
    trainer.save_freq=-1 \
    trainer.default_local_dir="${CKPTS_DIR}" \
    trainer.resume_mode=auto \
    trainer.nnodes="${NNODES}" \
    trainer.n_gpus_per_node="${n_gpus_training}" \
    rollout.nnodes="${NNODES}" \
    rollout.n_gpus_per_node="${n_gpus_rollout}" \
    rollout.total_rollout_steps="${total_rollout_steps}" \
    trainer.total_epochs=10 \
    trainer.test_freq="${test_freq}" \
    async_training.staleness_threshold="${staleness_threshold}" \
    async_training.trigger_parameter_sync_step="${trigger_parameter_sync_step}" \
    async_training.require_batches="${require_batches}" \
    async_training.partial_rollout="${partial_rollout}"

TRAIN_SCRIPT

training_pid=$!
echo "${training_pid}" >"${PID_FILE}"
disown "${training_pid}" 2>/dev/null || true

echo "Training started in the background."
echo "PID: ${training_pid}"
echo "Log: ${LOG_FILE}"
echo "PID file: ${PID_FILE}"

### 实时查看训练日志
如果希望持续查看新产生的日志，可以运行：

In [ ]:
%%bash

RUN_NAME="dapo_qwen2.5_7b_fully_async_2_2"
LOG_FILE="${HOME}/verl/logs/${RUN_NAME}.log"

if [[ ! -f "${LOG_FILE}" ]]; then
    echo "Log file does not exist: ${LOG_FILE}"
    exit 1
fi

tail -n 100 -f "${LOG_FILE}"

> **提示**
>
> 查看完成后，点击 Notebook 工具栏中的 **Stop（■）** 按钮，即可停止当前 cell 的日志输出。
>
> 该操作只会停止实时日志查看，不会停止后台运行的训练任务。需要继续查看时，重新运行上面的 cell 即可。

### 理解训练日志中的关键 Metrics
训练启动后，verl 会在每个 Training Step 输出一系列 metrics，用于反映 Fully Async Pipeline 的运行状态、Policy Update 的稳定性、Rollout Samples 的质量以及系统的资源利用率。下面我们将分类介绍其中较为重要的指标。

#### 训练进度与参数版本

| Metric | 示例值 | 含义 |
| --- | ---: | --- |
| `training/global_step` | `399` | Actor 已完成的累计 Training Steps |
| `training/epoch` | `0` | 当前所在的训练 epoch |
| `fully_async/count/current_param_version` | `99` | Rollout 侧当前使用的 Policy Parameter Version |

本次训练设置：

```text
trigger_parameter_sync_step = 4
```

也就是 Actor 每完成 4 个 Training Steps，向 Rollout Engines 同步一次最新参数。因此，当 `global_step=399` 时，`current_param_version=99` 与预期基本一致，说明参数同步正在正常推进。


#### Fully Async Pipeline 状态

| Metric | 示例值 | 含义 |
| --- | ---: | --- |
| `active_tasks_size` | `32` | Rollouter 当前正在执行的生成任务数 |
| `max_concurrent_samples` | `32` | Rollouter 允许同时处理的最大任务数 |
| `pending_queue_size` | `128` | 等待 Rollouter 处理的任务数 |
| `mq_queue_size` | `159.75` | MessageQueue 中等待 Trainer 消费的 samples 数量 |
| `required_samples` | `128` | Trainer 每次执行训练需要取得的 samples 数量 |
| `total_generated_samples` | `50943` | Rollouter 累计生成的 samples 数量 |
| `total_wait_time` | `51.23 s` | Trainer 等待训练 samples 的累计时间 |

本次训练配置为：

```text
require_batches    = 4
ppo_mini_batch_size = 32
```

因此，Trainer 每次训练需要获取：

```text
required_samples
= require_batches × ppo_mini_batch_size
= 4 × 32
= 128
```

当前 `mq_queue_size` 大于 `required_samples`，说明 MessageQueue 中有足够的数据可供 Trainer 消费。`active_tasks_size` 也达到了 `max_concurrent_samples`，说明 Rollouter 正在充分使用允许的生成并发。

`mq_queue_size` 等指标可能经过多个 Worker 的聚合或平均，因此日志中可能显示为小数。

#### Rollout Processing Time

| Metric | 示例值 | 含义 |
| --- | ---: | --- |
| `processing_time/avg` | `10.87 s` | Rollout 任务的平均处理时间 |
| `processing_time/tp50` | `10.02 s` | 50% 的任务在该时间内完成 |
| `processing_time/tp95` | `19.82 s` | 95% 的任务在该时间内完成 |
| `processing_time/tp99` | `25.70 s` | 99% 的任务在该时间内完成 |
| `processing_time/max` | `96.07 s` | 最慢任务的处理时间 |

大部分 Rollout 任务可以在 20 秒左右完成，但最慢任务耗时约 96 秒，说明存在少量 long-tail samples。

这正是 Fully Async Training 试图缓解的问题：较短的 Rollout 任务可以持续产生 samples，不需要等待最慢的任务完成后才开始 Training。

#### Sample Staleness

Fully Async Training 允许 Trainer 使用一定比例由旧版本 Policy 生成的 samples：

| Metric | 示例值 | 含义 |
| --- | ---: | --- |
| `staleness_threshold` | `0.1` | 允许使用的 stale samples 最大比例 |
| `staleness_samples` | `352.75` | 当前统计窗口中被计为 stale 的 samples |
| `stale_trajectory_processed` | `80784` | Trainer 累计处理的 stale trajectories 数量 |
| `dropped_stale_samples` | `0` | 因 freshness 限制而被丢弃的 stale samples 数量 |

一个 sample 会生成 `rollout.n=16` 条 trajectories，因此：

```text
stale_trajectory_processed
```

统计的是 trajectories，而不是原始 prompts。该指标还是累计值，随着训练进行持续增加是正常现象，不能仅根据它的绝对值判断 staleness 是否过高。

当前 `dropped_stale_samples=0`，说明没有 samples 因超过 freshness 限制而被丢弃。判断 staleness 是否影响训练时，应同时观察 validation accuracy、reward 和 Policy KL 的变化。

根据 verl 的 Fully Async 设计，`staleness_threshold` 越大，Rollout 和 Training 越容易保持并行，但 Trainer 也可能使用更多旧版本 Policy 生成的数据；因此需要在训练吞吐和 Policy Freshness 之间取得平衡。[verl Fully Async Policy 文档](https://github.com/verl-project/verl/blob/main/docs/advance/fully_async.md)


#### Policy Update 稳定性

| Metric | 示例值 | 含义 |
| --- | ---: | --- |
| `actor/pg_loss` | `0.01085` | Actor 的 Policy Gradient Loss |
| `actor/ppo_kl` | `0.00086` | 当前 Policy 与 Rollout Policy 之间的近似 KL divergence |
| `actor/pg_clipfrac` | `0.000995` | 触发 PPO clipping 的 token 比例 |
| `actor/pg_clipfrac_lower` | `0` | 触发 clipping lower bound 的比例 |
| `actor/grad_norm` | `0.233` | Actor 的 gradient norm |
| `actor/lr` | `1e-6` | 当前学习率 |

当前 `ppo_kl` 和 `pg_clipfrac` 都比较小，说明这一步的 Policy Update 幅度较为保守，大部分 probability ratios 没有触发 clipping。

`grad_norm=0.233` 低于配置的：

```text
grad_clip = 1.0
```

说明当前没有出现明显的 gradient explosion。

这些指标更适合结合趋势判断：

- `ppo_kl` 或 `pg_clipfrac` 突然大幅升高，可能表示 Policy Update 过大；
- `grad_norm` 持续异常升高，可能表示训练不稳定；
- 如果这些指标长期接近 0，同时 reward 和 validation accuracy 也没有提升，则可能表示 Policy Update 过于保守。

`pg_loss` 的正负和绝对值不能单独代表模型效果，应结合 reward、KL、gradient norm 和 validation metrics 一起分析。

#### Reward 与 Advantage

| Metric | 示例值 | 含义 |
| --- | ---: | --- |
| `critic/score/mean` | `-0.079` | 当前 Training Batch 的平均原始分数 |
| `critic/rewards/mean` | `-0.079` | 经过 Reward Manager 处理后的平均 reward |
| `critic/advantages/mean` | `-0.027` | Group Relative Advantage 的平均值 |
| `critic/advantages/max` | `3.75` | 当前 batch 中的最大 advantage |
| `critic/advantages/min` | `-3.75` | 当前 batch 中的最小 advantage |

`advantages/mean` 接近 0 符合 Group Relative Advantage 的预期，因为同一 Prompt 下的 Responses 会根据组内 reward 进行相对归一化。

`score/mean` 或 `rewards/mean` 的单步值不能直接代表模型最终效果。更重要的是观察它们是否随着训练逐渐提高，并结合 AIME 2024 的 validation accuracy 判断模型能力是否改善。


#### Validation Accuracy

`val-core/math_dapo/acc/mean@1` 是评估模型数学问题求解能力的核心验证指标：

| Metric | 含义 |
| --- | --- |
| `val-core/math_dapo/acc/mean@1` | 模型在 `math_dapo` 验证集上的平均准确率，每个 Prompt 使用 1 个 Response 进行评估 |

该指标可以拆分为：

- `val-core`：核心验证指标；
- `math_dapo`：使用的验证数据及 Reward Function；
- `acc`：答案准确率；
- `mean@1`：每个 Prompt 生成 1 个 Response，并计算所有验证样本的平均准确率。

例如：

```text
val-core/math_dapo/acc/mean@1 = 0.30
```

表示模型在当前验证中约有 **30%** 的问题回答正确。

#### Prompt 与 Response 长度

| Metric | 示例值 | 含义 |
| --- | ---: | --- |
| `prompt_length/mean` | `172` | Prompt 的平均 token 数 |
| `prompt_length/max` | `952` | 当前 batch 中最长的 Prompt |
| `response_length/mean` | `879` | Response 的平均 token 数 |
| `response_length/max` | `7881` | 当前 batch 中最长的 Response |
| `response_length/clip_ratio` | `0` | 因达到最大长度而被截断的 Response 比例 |
| `response/aborted_ratio` | `0` | 被中断的 Response 比例 |

当前最长 Response 为 7881 tokens，没有超过配置的：

```text
max_response_length = 8192
```

同时，`response_length/clip_ratio=0`，说明当前 batch 中没有 Response 因达到最大长度而被截断。

训练过程中需要重点观察：

- `response_length/mean` 是否持续快速增长；
- `response_length/clip_ratio` 是否逐渐升高；
- `response/aborted_ratio` 是否出现异常增长。

如果大量 Responses 接近 8192 tokens，可能说明模型开始生成过长的 reasoning paths，需要进一步观察 Overlong Reward Shaping 是否有效。

#### Step 时间与训练吞吐

| Metric | 示例值 | 含义 |
| --- | ---: | --- |
| `timing_s/gen` | `54.38 s` | Trainer 获取生成数据所花费的时间 |
| `timing_s/adv` | `0.11 s` | 计算 Advantage 的时间 |
| `timing_s/update_actor` | `574.71 s` | 更新 Actor 的时间 |
| `timing_s/param_sync` | `2.43 s` | 将最新 Policy 参数同步到 Rollout Engines 的时间 |
| `timing_s/step` | `631.65 s` | 完成当前 Training Step 的总时间 |
| `perf/total_num_tokens` | `8,612,699` | 当前 step 处理的 token 总数 |
| `perf/throughput` | `3408.82` | 框架统计的训练吞吐 |
